In [11]:
from pathlib import Path
import olefile
import zlib

# 입력 / 출력 폴더
INPUT_DIR = Path("./raw_files/hwp")
OUTPUT_DIR = Path("./extracted_images")

OUTPUT_DIR.mkdir(exist_ok=True)

# 이미지 확장자 판별
def detect_ext(data):

    if data.startswith(b"\xff\xd8"):
        return ".jpg"

    if data.startswith(b"\x89PNG"):
        return ".png"

    if data.startswith(b"GIF87a") or data.startswith(b"GIF89a"):
        return ".gif"

    if data.startswith(b"BM"):
        return ".bmp"

    return ".bin"


# HWP 이미지 추출 함수
def extract_images_from_hwp(hwp_path):

    print(f"\n처리 중: {hwp_path.name}")

    # Windows 안전 폴더명 처리
    safe_name = "".join(
        c for c in hwp_path.stem
        if c.isalnum() or c in ("_", "-")
    )

    output_subdir = OUTPUT_DIR / safe_name
    output_subdir.mkdir(parents=True, exist_ok=True)

    ole = olefile.OleFileIO(str(hwp_path))

    streams = ole.listdir()

    # 001부터 시작
    image_count = 1

    for stream in streams:

        stream_name = "/".join(stream)

        # BinData만 처리
        if "BinData" not in stream_name:
            continue

        try:

            data = ole.openstream(stream).read()

            # 압축 해제 시도
            try:
                data = zlib.decompress(data, -15)
            except:
                pass

            ext = detect_ext(data)

            # 이미지가 아닌 bin 파일 제외
            if ext == ".bin":
                print(f"건너뜀 (비이미지 데이터): {stream_name}")
                continue

            image_path = output_subdir / f"img_{image_count:03d}{ext}"

            with open(image_path, "wb") as f:
                f.write(data)

            print(f"저장 완료: {image_path.name}")

            image_count += 1

        except Exception as e:

            print(f"오류 발생: {stream_name} → {e}")

    print(f"총 {image_count - 1}개 이미지 추출 완료")


# 전체 HWP 처리
hwp_files = list(INPUT_DIR.glob("*.hwp"))

print(f"총 HWP 파일 수: {len(hwp_files)}")

for hwp_file in hwp_files:

    extract_images_from_hwp(hwp_file)

print("\n모든 작업 완료")

총 HWP 파일 수: 96

처리 중: (사)벤처기업협회_2024년 벤처확인종합관리시스템 기능 고도화 용역사업 .hwp
저장 완료: img_001.png
저장 완료: img_002.jpg
저장 완료: img_003.bmp
저장 완료: img_004.bmp
저장 완료: img_005.jpg
저장 완료: img_006.jpg
저장 완료: img_007.jpg
저장 완료: img_008.jpg
저장 완료: img_009.jpg
총 9개 이미지 추출 완료

처리 중: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp
저장 완료: img_001.png
저장 완료: img_002.png
저장 완료: img_003.png
저장 완료: img_004.png
저장 완료: img_005.bmp
저장 완료: img_006.bmp
저장 완료: img_007.bmp
저장 완료: img_008.png
저장 완료: img_009.png
저장 완료: img_010.png
저장 완료: img_011.png
저장 완료: img_012.bmp
저장 완료: img_013.png
저장 완료: img_014.jpg
저장 완료: img_015.png
저장 완료: img_016.jpg
저장 완료: img_017.png
총 17개 이미지 추출 완료

처리 중: (사）한국대학스포츠협의회_KUSF 체육특기자 경기기록 관리시스템 개발.hwp
저장 완료: img_001.png
저장 완료: img_002.bmp
저장 완료: img_003.bmp
저장 완료: img_004.bmp
저장 완료: img_005.bmp
저장 완료: img_006.jpg
총 6개 이미지 추출 완료

처리 중: (재)예술경영지원센터_통합 정보시스템 구축 사전 컨설팅.hwp
저장 완료: img_001.bmp
저장 완료: img_002.bmp
저장 완료: img_003.bmp
저장 완료: img_004.bmp
저장 완료: img_005.bmp
저장 완료: img_006.bmp
총 6개 이미지 추출 완료

In [11]:
from pathlib import Path
import hashlib

# PDF 이미지 추출에는 PyMuPDF가 필요합니다.
# 설치가 안 되어 있으면 아래 명령을 한 번 실행하세요.
# !pip install pymupdf
try:
    import fitz  # PyMuPDF
except ImportError as exc:
    raise ImportError("PyMuPDF가 설치되어 있지 않습니다. 노트북에서 `!pip install pymupdf`를 먼저 실행하세요.") from exc

# 입력 / 출력 폴더
PDF_INPUT_DIR = Path("./raw_files/pdf")
PDF_OUTPUT_DIR = Path("./extracted_images")

PDF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# OCR용 후보 이미지 필터 기준
MIN_WIDTH = 100
MIN_HEIGHT = 40
MIN_AREA = 10_000
MIN_BYTES = 1_000


def make_safe_name(file_stem):
    """HWP 추출 때와 같은 방식으로 문서 폴더명을 만듭니다."""
    return "".join(
        c for c in file_stem
        if c.isalnum() or c in ("_", "-")
    )


def should_keep_pdf_image(image_data, image_bytes, seen_hashes):
    width = image_data.get("width", 0)
    height = image_data.get("height", 0)
    area = width * height
    byte_size = len(image_bytes)
    image_hash = hashlib.md5(image_bytes).hexdigest()

    if image_hash in seen_hashes:
        return False, "중복 이미지"

    if width < MIN_WIDTH or height < MIN_HEIGHT:
        return False, f"작은 이미지({width}x{height})"

    if area < MIN_AREA:
        return False, f"작은 면적({width}x{height})"

    if byte_size < MIN_BYTES:
        return False, f"작은 파일({byte_size} bytes)"

    seen_hashes.add(image_hash)
    return True, ""


def extract_images_from_pdf(pdf_path):
    print(f"\n처리 중: {pdf_path.name}")

    source_doc_key = make_safe_name(pdf_path.stem)
    output_subdir = PDF_OUTPUT_DIR / source_doc_key
    output_subdir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)
    image_count = 1
    skipped_count = 0
    seen_hashes = set()

    for page_index in range(len(doc)):
        page = doc[page_index]
        image_list = page.get_images(full=True)

        if not image_list:
            print(f"페이지 {page_index + 1}: 이미지 없음")
            continue

        for image_info in image_list:
            xref = image_info[0]

            try:
                image_data = doc.extract_image(xref)
                image_bytes = image_data["image"]
                keep, reason = should_keep_pdf_image(image_data, image_bytes, seen_hashes)

                if not keep:
                    skipped_count += 1
                    print(f"건너뜀: page {page_index + 1}, xref {xref} - {reason}")
                    continue

                ext = image_data.get("ext", "png").lower()

                if ext == "jpeg":
                    ext = "jpg"

                image_path = output_subdir / f"img_{image_count:03d}.{ext}"

                with open(image_path, "wb") as f:
                    f.write(image_bytes)

                width = image_data.get("width", 0)
                height = image_data.get("height", 0)
                print(f"저장 완료: {image_path.name} ({width}x{height}, page {page_index + 1}, xref {xref})")
                image_count += 1

            except Exception as e:
                print(f"오류 발생: page {page_index + 1}, xref {xref} -> {e}")

    doc.close()
    print(f"저장 {image_count - 1}개, 건너뜀 {skipped_count}개")


pdf_files = sorted(PDF_INPUT_DIR.glob("*.pdf"))

print(f"총 PDF 파일 수: {len(pdf_files)}")

for pdf_file in pdf_files:
    extract_images_from_pdf(pdf_file)

print("\nPDF 이미지 추출 완료")

총 PDF 파일 수: 4

처리 중: 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
건너뜀: page 1, xref 18 - 작은 면적(141x40)
저장 완료: img_001.png (416x117, page 1, xref 38)
건너뜀: page 2, xref 18 - 작은 면적(141x40)
건너뜀: page 3, xref 18 - 작은 면적(141x40)
건너뜀: page 4, xref 18 - 작은 면적(141x40)
건너뜀: page 5, xref 18 - 작은 면적(141x40)
건너뜀: page 6, xref 18 - 작은 면적(141x40)
건너뜀: page 7, xref 18 - 작은 면적(141x40)
건너뜀: page 8, xref 18 - 작은 면적(141x40)
저장 완료: img_002.jpg (901x496, page 8, xref 72)
건너뜀: page 9, xref 18 - 작은 면적(141x40)
저장 완료: img_003.jpg (878x512, page 9, xref 80)
건너뜀: page 10, xref 18 - 작은 면적(141x40)
저장 완료: img_004.png (1035x552, page 10, xref 83)
건너뜀: page 11, xref 18 - 작은 면적(141x40)
건너뜀: page 12, xref 18 - 작은 면적(141x40)
건너뜀: page 13, xref 18 - 작은 면적(141x40)
건너뜀: page 14, xref 18 - 작은 면적(141x40)
건너뜀: page 15, xref 18 - 작은 면적(141x40)
저장 완료: img_005.png (538x255, page 15, xref 94)
건너뜀: page 16, xref 18 - 작은 면적(141x40)
저장 완료: img_006.png (1115x710, page 16, xref 97)
건너뜀: page 17, xref 18 - 작은 면적(141x40)
저장 완료: img_007.jpg (538x255, p